# Model Calibration in Python: The Metric Decides the Method

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/supervised/model_calibration.ipynb)

A classifier that says 0.9 should be right nine times in ten. Accuracy cannot tell you whether yours is, and neither can AUC, precision or F1, because all of them depend on the ordering of the scores rather than on their values.

This notebook measures how five model families miscalibrate, in which direction, and what three standard repairs do about it. Then it shows the part most treatments leave out: the metric you choose to judge calibration by decides which repair you pick, and the popular one picks wrong.

Everything runs on a CPU in two to three minutes. No GPU, no framework beyond scikit-learn.

Companion post: [Model Calibration in Python: The Metric Decides the Method](https://sesen.ai/blog/model-calibration-python)

## 1. A dataset and three splits

Forest cover type reduced to the largest class against the rest, which is the binary problem Niculescu-Mizil and Caruana used in 2005.

Three disjoint splits, and the separation is the whole point: the model trains on the first, the calibrator fits on the second, and every number reported comes off the third. Section 7 shows what happens when you skip that.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import fetch_covtype
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

N_TRAIN, N_CAL, N_TEST = 5000, 2000, 20000


def load(seed=0):
    """Forest cover type, reduced to the largest class against the rest.

    Three disjoint splits. The middle one is the calibration set, and the whole
    post depends on the model never having seen it.
    """
    d = fetch_covtype()
    X, y = d.data, (d.target == 2).astype(int)
    idx = np.random.default_rng(seed).permutation(len(y))[: N_TRAIN + N_CAL + N_TEST]
    tr, ca, te = idx[:N_TRAIN], idx[N_TRAIN : N_TRAIN + N_CAL], idx[N_TRAIN + N_CAL :]
    scaler = StandardScaler().fit(X[tr])
    return tuple((scaler.transform(X[i]), y[i]) for i in (tr, ca, te))


(Xtr, ytr), (Xca, yca), (Xte, yte) = load()
print(f"train {len(ytr)}, calibration {len(yca)}, test {len(yte)}")
print(f"positive rate {ytr.mean():.3f}")

## 2. The reliability diagram, and the number under it

Sort the predictions into bins, and in each bin compare the average prediction with the fraction that turned out positive. Everything else here is a way of summarising that table into one number, and every summary throws something away.

ECE throws away the direction, because it takes an absolute value. The calibration slope recovers it: refit the labels against the model's own log-odds and read the coefficient. Below 1 the scores are too extreme, above 1 too timid.

In [ ]:
def reliability(p, y, bins=15):
    """Per-bin (mean predicted probability, observed fraction, count).

    This is the whole of calibration measurement. Everything below is a way of
    summarising these three columns into one number, and every one of those
    summaries throws something away.
    """
    edges = np.linspace(0, 1, bins + 1)
    edges[0], edges[-1] = -np.inf, np.inf
    rows = [(p[m].mean(), y[m].mean(), m.sum())
            for lo, hi in zip(edges[:-1], edges[1:])
            if (m := (p > lo) & (p <= hi)).sum()]
    return np.array(rows)


def ece(p, y, bins=15):
    """Expected calibration error: the diagram's gaps, weighted by bin size."""
    r = reliability(p, y, bins)
    return float((r[:, 2] / r[:, 2].sum() * np.abs(r[:, 1] - r[:, 0])).sum())


def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


def calibration_slope(p, y):
    """Refit y ~ sigmoid(a * logit(p) + b) and return a.

    Below 1 the scores are too extreme, which is overconfidence. Above 1 they
    are too timid. This is the direction that ECE, being an absolute value,
    cannot tell you.
    """
    return float(LogisticRegression(max_iter=1000)
                 .fit(logit(p).reshape(-1, 1), y).coef_[0, 0])


def plot_reliability(ax, p, y, colour, title):
    r = reliability(p, y)
    ax.plot([0, 1], [0, 1], "--", color="#8a8a8a", lw=1.1)
    ax.plot(r[:, 0], r[:, 1], "o-", color=colour, lw=2, ms=4)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_title(title, fontsize=10)
    ax.text(0.04, 0.93, f"ECE {ece(p, y):.3f}\nslope {calibration_slope(p, y):.2f}",
            fontsize=9, va="top", weight="bold", color=colour)

## 3. Five families, five shapes

Watch two columns in particular. The random forest is the most accurate model here and among the worst calibrated; logistic regression is the least accurate and the best calibrated. Accuracy and calibration are different axes.

Note the gradient boosting slope. Boosted trees are widely described as overconfident, and they come back timid, which is what the 2005 paper found too.

The SVM is a different problem: it has no probability to return, only a distance to the boundary, so the code has to invent one.

In [ ]:
def scores(model, name, Xs):
    """An SVM has no probability to return, only a distance to the boundary."""
    if name == "SVM (RBF)":
        d = [model.decision_function(X) for X in Xs]
        lo, hi = d[0].min(), d[0].max()
        return [np.clip((di - lo) / (hi - lo), 0, 1) for di in d]
    return [model.predict_proba(X)[:, 1] for X in Xs]


FAMILIES = {
    "logistic regression": LogisticRegression(max_iter=1000),
    "random forest": RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1),
    "gradient boosting": GradientBoostingClassifier(random_state=0),
    "small MLP": MLPClassifier((64, 32), max_iter=400, random_state=0),
    "SVM (RBF)": SVC(random_state=0),
}

fitted = {}
fig, axes = plt.subplots(1, 5, figsize=(16, 3.4), sharey=True)
for ax, (name, model) in zip(axes, FAMILIES.items()):
    model.fit(Xtr, ytr)
    pc, pt = scores(model, name, [Xca, Xte])
    fitted[name] = (pc, pt)
    a = calibration_slope(pt, yte)
    plot_reliability(ax, pt, yte, "#d99120" if a < 0.9 else "#1f9e9e", name)
    ax.set_xlabel("predicted probability")
    print(f"{name:<22} accuracy {((pt >= 0.5) == yte).mean():.4f}   "
          f"AUC {roc_auc_score(yte, pt):.4f}   ECE {ece(pt, yte):.4f}   slope {a:.3f}")
axes[0].set_ylabel("observed fraction positive")
plt.show()

## 4. Three repairs

Each fits a one-dimensional function from the score to a probability, on held-out data. Platt scaling has a slope and an intercept. Temperature scaling is Platt with the intercept deleted, which is why it can fix confidence but not an offset. Isotonic regression is the best non-decreasing step function, with no shape assumption at all.

The last line prints the identity worth remembering: the fitted temperature is one over the calibration slope.

In [ ]:
def platt(pc, yc, pt):
    """One logistic regression on the score. Strictly monotone, two parameters."""
    return (LogisticRegression().fit(pc.reshape(-1, 1), yc)
            .predict_proba(pt.reshape(-1, 1))[:, 1])


def isotonic(pc, yc, pt):
    """The best non-decreasing step function. Weakly monotone, unbounded flexibility."""
    return IsotonicRegression(out_of_bounds="clip").fit(pc, yc).predict(pt)


def temperature(pc, yc, pt, grid=np.linspace(0.05, 10, 800)):
    """Divide the logit by one number, picked by log loss on held-out data.

    Platt with the intercept deleted, so it can correct a slope but not an
    offset. That is the whole difference, and it shows up below.
    """
    zc, zt = logit(pc), logit(pt)
    def nll(T):
        q = np.clip(1 / (1 + np.exp(-zc / T)), 1e-15, 1 - 1e-15)
        return -np.mean(yc * np.log(q) + (1 - yc) * np.log(1 - q))
    T = float(grid[np.argmin([nll(t) for t in grid])])
    return 1 / (1 + np.exp(-zt / T)), T


pc, pt = fitted["small MLP"]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), sharey=True)
plot_reliability(axes[0], pt, yte, "#d99120", "no recalibration")
plot_reliability(axes[1], platt(pc, yca, pt), yte, "#1f9e9e", "Platt scaling")
pT, T = temperature(pc, yca, pt)
plot_reliability(axes[2], pT, yte, "#1f9e9e", f"temperature scaling (T = {T:.2f})")
plot_reliability(axes[3], isotonic(pc, yca, pt), yte, "#1f9e9e", "isotonic regression")
for ax in axes:
    ax.set_xlabel("predicted probability")
axes[0].set_ylabel("observed fraction positive")
plt.show()
print(f"fitted T = {T:.3f}, and 1 / calibration slope = {1 / calibration_slope(pt, yte):.3f}")

## 5. The ranking never moves

Platt and temperature scaling are strictly increasing functions of the score, so they cannot reorder two rows, and AUC depends on nothing but that order.

Isotonic regression is only weakly monotone. Its steps flatten rows that used to be ordered, and every tie costs a little discrimination. If a recalibration method moves your AUC noticeably, it is refitting rather than recalibrating.

In [ ]:
print(f"{'family':<22}{'AUC raw':>10}{'Platt':>10}{'temperature':>13}{'isotonic':>11}")
for name, (pc, pt) in fitted.items():
    row = [roc_auc_score(yte, q) for q in
           (pt, platt(pc, yca, pt), temperature(pc, yca, pt)[0], isotonic(pc, yca, pt))]
    print(f"{name:<22}" + "".join(f"{v:>10.5f}" if i < 2 else f"{v:>{13 if i == 2 else 11}.5f}"
                                  for i, v in enumerate(row)))
print("\nPlatt and temperature are strictly monotone, so they cannot reorder anything.")
print("Isotonic is only weakly monotone: it creates ties, and ties cost AUC.")

## 6. Where the metrics disagree

Isotonic regression won on ECE for every family above. The standard advice is that it overfits without a lot of calibration data, so shrink the calibration set and watch it fail.

It does fail, and ECE cannot see it. Read the two 'picks' columns against each other, then read the last column, which is the answer: isotonic is sending a large share of test predictions to exactly 0 or exactly 1. ECE's per-bin term is bounded by 1 no matter how wrong a prediction is, so a confident error costs it nothing extra. Log loss is unbounded on exactly that.

Choose the method on a proper scoring rule and use ECE afterwards to describe what you got.

In [ ]:
def log_loss(p, y):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


pc, pt = fitted["small MLP"]
print(f"{'n_cal':>7}{'ECE iso':>10}{'ECE Platt':>11}{'  ECE picks':>13}"
      f"{'LL iso':>10}{'LL Platt':>10}{'  log loss picks':>17}{'iso at 0 or 1':>15}")
for n in (50, 100, 200, 500, 1000, 2000):
    qi, qp = isotonic(pc[:n], yca[:n], pt), platt(pc[:n], yca[:n], pt)
    ei, ep, li, lp = ece(qi, yte), ece(qp, yte), log_loss(qi, yte), log_loss(qp, yte)
    sat = np.mean((qi <= 1e-12) | (qi >= 1 - 1e-12))
    print(f"{n:>7}{ei:>10.4f}{ep:>11.4f}{'isotonic' if ei < ep else 'Platt':>13}"
          f"{li:>10.4f}{lp:>10.4f}{'isotonic' if li < lp else 'Platt':>17}{sat:>14.1%}")

## 7. Calibrate on data the model has never seen

The rule is easy to state and easy to skip, and skipping it produces no warning. It produces a perfect reliability diagram.

The forest has near-perfect predictions on rows it memorised, so a calibrator fitted there learns to map them straight onto the labels. Scored on those same rows it is flawless. Taken to held-out data it is worse than doing nothing at all.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(Xtr, ytr)
p_tr, p_ca, p_te = (rf.predict_proba(X)[:, 1] for X in (Xtr, Xca, Xte))

wrong = isotonic(p_tr, ytr, p_te)      # calibrator fitted on rows the forest memorised
right = isotonic(p_ca, yca, p_te)      # calibrator fitted on rows it never saw

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
plot_reliability(axes[0], isotonic(p_tr, ytr, p_tr), ytr, "#1f9e9e",
                 "in-sample, scored in-sample")
plot_reliability(axes[1], wrong, yte, "#b4483c", "in-sample calibrator, held-out rows")
plot_reliability(axes[2], right, yte, "#1f9e9e", "held-out calibrator, held-out rows")
for ax in axes:
    ax.set_xlabel("predicted probability")
axes[0].set_ylabel("observed fraction positive")
plt.show()

for label, q in (("do nothing", p_te), ("calibrate in-sample", wrong),
                 ("calibrate held-out", right)):
    print(f"{label:<22} test ECE {ece(q, yte):.4f}   test log loss {log_loss(q, yte):.4f}")

## Exercises

1. **Break the split.** Fit Platt scaling instead of isotonic on the training rows in section 7. It degrades far less. Work out why the two-parameter method survives the mistake that destroys the non-parametric one.
2. **Find your own crossover.** Sweep the calibration set size for the random forest rather than the MLP, on log loss. The forest starts nearly calibrated, so the crossover moves. Where to?
3. **Break the metric on purpose.** Take an isotonic fit from 200 rows and clip its output to [0.01, 0.99]. Log loss improves sharply and ECE barely moves. That gap is the confident-wrongness ECE cannot price.
4. **Multi-class.** Extend `ece` to Guo et al.'s definition on `max(p)` against accuracy and rerun on the full seven-class cover type problem. Which families keep their direction?
5. **The imbalance shift.** Subsample the positive class to 5%, oversample back to 50/50, and measure how far the logistic regression's intercept moves. Compare it with `log((0.5/0.5) / (pi/(1-pi)))`, and then try the same correction on the random forest and watch it fail.


## Further reading

- Platt (1999), [Probabilistic Outputs for Support Vector Machines](https://www.cs.colorado.edu/~mozer/Teaching/syllabi/6622/papers/Platt1999.pdf)
- Zadrozny & Elkan (2002), [Transforming Classifier Scores into Accurate Multiclass Probability Estimates](https://cseweb.ucsd.edu/~elkan/calibrated.pdf)
- Niculescu-Mizil & Caruana (2005), [Predicting Good Probabilities with Supervised Learning](https://www.cs.cornell.edu/~alexn/papers/calibration.icml05.crc.rev3.pdf)
- Guo, Pleiss, Sun & Weinberger (2017), [On Calibration of Modern Neural Networks](https://arxiv.org/abs/1706.04599)
- Murphy (1973), [A New Vector Partition of the Probability Score](https://journals.ametsoc.org/view/journals/apme/12/4/1520-0450_1973_012_0595_anvpot_2_0_co_2.xml)
- [SMOTE vs Class Weights: A Guide to Class Imbalance](https://sesen.ai/blog/class-imbalance-downsampling-smote-comprehensive-guide), the resampling that shifts every probability by a known amount
